# Synthesis and Analysis of Suspensions and Exclusions Data


## Table of contents 


- [Background](#background)


## Background


This report will cover how to synthesise a dataset that is used in my organisation. This data is a timeseries one about suspensions and exclusions in England. 


It includes the following grographical breakdowns:


- national


- regional


- local authority




It also includes the following education phases breakdowns:


- primary


- secondary


- special. 


The data contains metrics for each breakdown about: 


- Headcounts - Number of pupils for the relevant breakdown


- Suspensions - Number of pupils suspended 


- Exclusions - Number of pupils excluded 


- Suspension rate - The rate of suspensions calculated by suspensions/headcounts for each relevant breakdown.


- Exlcusions rate - The rate of exclusions calculated by exclusions/headcounts for each relevant breakdown.


## Generating the data - Summary of approach 




The data generation is made reprodicble by using seeding functions.


The synthetic dataset is generated at the local authority level for different time periods and education phases. For each local authority and phase:


- Headcounts are generated using a log-normal distribution to simulate realistic pupil numbers.
- Suspensions are generated using a negative binomial distribution, ensuring the value is always less than the headcount.
- Exclusions are generated using a Poisson distribution, also clipped to be less than the headcount.
- All values are clipped to avoid unrealistic results (e.g., negative or excessively large numbers).


After generating the local authority-level data, aggregation is performed to calculate totals for headcounts, suspensions, and exclusions at regional and national levels. Suspension and exclusion rates are then calculated for each breakdown.

## Generating the data - step by step 

## Importing needed packages 

First we start by importing the packages needed 

### Step 1: Import required libraries

In [ ]:
import pandas as pd
import numpy as np

### Step 2: Set up parameters and random seed

In [ ]:
np.random.seed(42)
education_phases = ["primary", "secondary", "special"]
time_periods = ["202324", "202425"]
geographic_levels = ["National", "Regional", "Local Authority"]

# Example local authorities and regions for demonstration
local_authorities = [
    {"la_code": "E06000001", "name": "Hartlepool", "parent_geography_code": "E12000001"},
    {"la_code": "E06000002", "name": "Middlesbrough", "parent_geography_code": "E12000001"}
]
region_dict = {"E12000001": "North East"}
country_code = "E92000001"
country_name = "England"

### Step 3: Generate synthetic data for each local authority and education phase

In [ ]:
main_data = []
for la_info in local_authorities:
    la_code = la_info["la_code"]
    parent_region_code = la_info["parent_geography_code"]
    la_name = la_info["name"]
    region_name = region_dict[parent_region_code]
    for phase in education_phases:
        headcount = int(np.clip(np.random.lognormal(1e3, 4e3), 1e2, 3e4))
        suspensions = int(np.clip(np.random.negative_binomial(50, 0.05), 0, min(1000, headcount-1)))
        exclusions = int(np.clip(np.random.poisson(20), 0, headcount-1))
        main_data.append({
            "time_identifier": "Autumn Term",
            "time_period": time_periods[0],
            "geographic_level": "Local Authority",
            "country_code": country_code,
            "country_name": country_name,
            "region_code": parent_region_code,
            "region_name": region_name,
            "la_code": la_code,
            "la_name": la_name,
            "education_phases": phase,
            "headcount": headcount,
            "suspensions": suspensions,
            "exclusions": exclusions
        })

### Step 4: Create a DataFrame from the generated data

In [ ]:
df = pd.DataFrame(main_data)
df.head()

### Step 5: Calculate rates

In [ ]:
df["susp_rate"] = df["suspensions"] / df["headcount"]
df["excl_rate"] = df["exclusions"] / df["headcount"]
df.head()

### Step 6: Save the synthetic data to CSV

In [ ]:
df.to_csv("example.csv", index=False)
print("example.csv has been created.")